# Imports

In [29]:
source("local-functions.R")
library(ggplot2)

# Study 1

## What the non-participants said

In [30]:
df1 = haven::read_sav("../data/Pilot1.sav") |> 
    add_participation() |> 
    dplyr::filter(participation %in% c("Didn't participate","Didn't answer the question"))

# Source column names
variables <- c(paste0("G08Q43_SQ", sprintf("%03d", 1:11)), "G08Q43_other")

# Descriptive labels
labels <- c(
  "Concern about the possible discomfort associated with the procedures",
  "Concern about possible health risk associated with procedures",
  "Having to travel too far to take part in procedures",
  "Financial reasons",
  "Time constraints",
  "Unwillingness to have my H. pylori infection treated",
  "Belief in my good health and the unnecessity of these procedures",
  "Mistrust towards the healthcare system/ my healthcare provider",
  "Do not understand the methodology of the study",
  "Do not understand the purpose of the study",
  "Not seeing a benefit to participation",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df1$G08Q43_other[
  !is.na(df1$G08Q43_other) &
    trimws(df1$G08Q43_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df1[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why wouldn’t you accept to participate in the study?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why wouldn’t you accept to participate in the study?

,Time constraints,Other,Concern about the possible discomfort associated with the procedures,Belief in my good health and the unnecessity of these procedures,Concern about possible health risk associated with procedures,Not seeing a benefit to participation,Having to travel too far to take part in procedures,Mistrust towards the healthcare system/ my healthcare provider,Do not understand the purpose of the study,Financial reasons,Unwillingness to have my H. pylori infection treated,Do not understand the methodology of the study
Time constraints,19,0,0,2,0,1,1,0,0,0,0,0
Other,0,8,0,0,0,0,0,0,0,0,0,0
Concern about the possible discomfort associated with the procedures,0,0,6,0,1,0,0,0,0,0,0,0
Belief in my good health and the unnecessity of these procedures,2,0,0,3,0,0,0,1,1,0,0,0
Concern about possible health risk associated with procedures,0,0,1,0,2,0,0,0,0,0,0,0
Not seeing a benefit to participation,1,0,0,0,0,2,0,0,0,0,0,0
Having to travel too far to take part in procedures,1,0,0,0,0,0,1,0,0,0,0,0
Mistrust towards the healthcare system/ my healthcare provider,0,0,0,1,0,0,0,1,1,0,0,0
Do not understand the purpose of the study,0,0,0,1,0,0,0,1,1,0,0,0
Financial reasons,0,0,0,0,0,0,0,0,0,0,0,0


## Data

In [31]:
df1 = haven::read_sav("../data/Pilot1.sav")
df1 = df1 |> 
    add_participation() |>
    add_gender() |> dplyr::filter(gender %in% c("Male", "Female")) |>
    add_education() |> dplyr::filter(!(education %in% c("I prefer not to answer", "Didn't answer the question"))) |>
    add_health() |> dplyr::filter(!(health %in% c("Very bad","I don't know","I prefer not to respond","Didn't answer the question"))) |>
    add_family_history(1) |> dplyr::filter(!(family_history %in% c("I prefer not to answer", "Didn't answer the question"))) |>
    add_knowing() |> dplyr::filter(!(knowing %in% c("Didn't answer the question"))) |> 
    add_prior_pilory() |>
    add_awareness_pilory() |> dplyr::filter(!(awareness_pilory %in% c("I prefer not to answer", "Didn't answer the question"))) |>
                                dplyr::mutate(awareness_pilory = forcats::fct_collapse(awareness_pilory,"No" = c("No", "I don't know"))) |> 
    add_awareness_GC(1) |> dplyr::filter(!(awareness_GC %in% c("I prefer not to answer", "Didn't answer the question"))) |> 
                                dplyr::mutate(awareness_GC = forcats::fct_collapse(awareness_GC,"No" = c("No", "I don’t know")))

In [32]:
print("----------------awareness_GC---------------------")
print(summary(df1$awareness_GC))
print("--------------awareness_pilory-------------------")
print(summary(df1$awareness_pilory))
print("----------------prior_pilory---------------------")
print(summary(df1$prior_pilory))
print("---------------knowing---------------------------")
print(summary(df1$knowing))
print("--------------family_history---------------------")
print(summary(df1$family_history))
print("----------------health---------------------------")
print(summary(df1$health))
print("---------------education-------------------------")
print(summary(df1$education))
print("-----------------gender---------------------------")
print(summary(df1$gender))
print("----------------participation---------------------")
print(summary(df1$participation))

[1] "----------------awareness_GC---------------------"
                        No                        Yes 
                       463                        234 
    I prefer not to answer Didn't answer the question 
                         0                          0 
[1] "--------------awareness_pilory-------------------"
                       Yes                         No 
                       501                        196 
    I prefer not to answer Didn't answer the question 
                         0                          0 
[1] "----------------prior_pilory---------------------"
                        No                      Other 
                       217                         27 
              I don’t know     I prefer not to answer 
                         6                          0 
Didn't answer the question 
                       447 
[1] "---------------knowing---------------------------"
                        No                        Yes 
     

## Linear Model

In [33]:
# Fit linear regression
model_participation <- stats::lm(
  participation ~ gender + education + health + family_history + knowing + prior_pilory + awareness_pilory + awareness_GC,
  data = df1 |> dplyr::mutate(participation = dplyr::case_when(
      participation == "Participated" ~ 1,
      TRUE ~ 0)),
      contrasts = list(
        gender = "contr.treatment",
        education = "contr.treatment",
        health = "contr.poly",
        family_history = "contr.treatment",
        knowing = "contr.treatment",
        prior_pilory = "contr.treatment",
        awareness_pilory = "contr.treatment",
        awareness_GC = "contr.treatment"
      )
)
# model_participation = broom::tidy(model_participation)

### Fitted values

In [34]:
# Model summary
broom::augment(model_participation)

participation,gender,education,health,family_history,knowing,prior_pilory,awareness_pilory,awareness_GC,.fitted,.resid,.hat,.sigma,.cooksd,.std.resid
<dbl>,<fct>,<ord>,<ord>,<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,Female,Higher education (bachelor’s degree or higher),Excellent,No,Yes,Didn't answer the question,Yes,Yes,0.9600320,0.0399679757,0.011501774,0.2300583,2.223619e-05,0.174862103
1,Female,Higher education (bachelor’s degree or higher),Good,No,No,No,Yes,No,0.9679474,0.0320526212,0.013260044,0.2300601,1.654587e-05,0.140356873
1,Female,Higher education (bachelor’s degree or higher),Excellent,Other,Yes,Didn't answer the question,Yes,Yes,0.9928907,0.0071093299,0.009939662,0.2300633,6.060782e-07,0.031079164
1,Female,Higher education (bachelor’s degree or higher),Excellent,Other,No,No,Yes,No,0.9887335,0.0112664799,0.013844035,0.2300630,2.136835e-06,0.049349975
1,Male,Higher education (bachelor’s degree or higher),Good,No,I don’t know,Didn't answer the question,Yes,Yes,0.9594871,0.0405129370,0.024940724,0.2300581,5.091637e-05,0.178463625
1,Male,Higher education (bachelor’s degree or higher),Good,No,No,Didn't answer the question,Yes,Yes,0.9454608,0.0545391753,0.013049458,0.2300538,4.712396e-05,0.238798922
1,Male,High school or vocational,Good,I don't know,Yes,Didn't answer the question,Yes,Yes,0.7496312,0.2503687515,0.036605259,0.2298554,2.923602e-03,1.109556644
1,Female,Higher education (bachelor’s degree or higher),Excellent,Other,No,Didn't answer the question,Yes,Yes,0.9771166,0.0228834213,0.011583891,0.2300617,7.342429e-06,0.100120391
1,Female,High school or vocational,Good,Other,No,No,Yes,No,0.8381233,0.1618767425,0.022318006,0.2299778,7.235225e-04,0.712126485


### Coefficients

In [35]:
results_participation <- broom::tidy(
  model_participation,
  exponentiate = FALSE,
  conf.int = TRUE
) |>
  dplyr::select(
    term,
    estimate,
    conf.low,
    conf.high,
    p.value
  ) |>
  dplyr::rename(
    Variable = term,
    CI_lower = conf.low,
    CI_upper = conf.high,
    p_value = p.value
  )

results_participation

Variable,estimate,CI_lower,CI_upper,p_value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),1.023611149,0.94162833,1.105593971,1.262233e-95
genderMale,-0.010869613,-0.04755582,0.025816598,5.609317e-01
educationHigh school or vocational,-0.162682767,-0.21341745,-0.111948088,5.477002e-10
educationPrimary/did not finish high school,-0.226071995,-0.39936068,-0.052783307,1.063583e-02
health.L,0.069792079,-0.10848412,0.248068281,4.423626e-01
health.Q,0.028621822,-0.10812250,0.165366150,6.812249e-01
health.C,0.010601601,-0.06362322,0.084826424,7.792237e-01
family_historyNo,-0.032858646,-0.06909360,0.003376303,7.543945e-02
family_historyI don't know,-0.081779546,-0.15603797,-0.007521121,3.094116e-02


## Separation models

### Checking problematic variables

In [36]:
model_separation <- stats::glm(
participation ~ gender + education + family_history + knowing + awareness_pilory + awareness_GC + prior_pilory + health,
  data = df1,
  family = stats::binomial(link = "logit"),
  contrasts = list(
    gender = "contr.treatment",
    education = "contr.treatment",
    family_history = "contr.treatment",
    knowing = "contr.treatment",
    awareness_pilory = "contr.treatment",
    awareness_GC = "contr.treatment",
    prior_pilory = "contr.treatment",
    health = "contr.poly"
  ),
  method = detectseparation::detect_separation
)

model_separation

Implementation: ROI | Solver: lpsolve 
Separation: TRUE 
Existence of maximum likelihood estimates
                                (Intercept) 
                                       -Inf 
                                 genderMale 
                                          0 
         educationHigh school or vocational 
                                          0 
educationPrimary/did not finish high school 
                                          0 
                           family_historyNo 
                                          0 
                 family_historyI don't know 
                                          0 
                                 knowingYes 
                                          0 
                       knowingI don’t know  
                                          0 
                         awareness_piloryNo 
                                          0 
                            awareness_GCYes 
                                          0 
 

Variables exhibiting complete or quasi-complete separation were investigated. Despite attempts to address separation using bias-reduced logistic regression, numerical convergence could not be achieved. These variables were therefore excluded from the final multivariable model.

check the three-way table participation, awareness_pilory and family_history

In [37]:
model_participation_br <- stats::glm(
  participation ~ health + prior_pilory,
  data = df1,
  family = stats::binomial(link = "logit"),
  contrasts = list(
    health = "contr.poly",
    prior_pilory = "contr.treatment"
  ),
  method = brglm2::brglmFit
)

Warning message:
“brglmFit: algorithm did not converge. Try changing the optimization algorithm defaults, e.g. the defaults for one or more of `maxit`, `epsilon`, `slowit`, and `response_adjustment`; see `?brglm_control` for default values and available options”
Warning message:
“brglmFit: fitted probabilities numerically 0 or 1 occurred”


In [38]:
model_participation <- stats::glm(
  participation ~ health + prior_pilory,
  data = df1,
  family = stats::binomial(link = "logit"),
  contrasts = list(
    health = "contr.poly",
    prior_pilory = "contr.treatment"
  )
)
summary(model_participation)


Call:
stats::glm(formula = participation ~ health + prior_pilory, family = stats::binomial(link = "logit"), 
    data = df1, contrasts = list(health = "contr.poly", prior_pilory = "contr.treatment"))

Coefficients:
                                       Estimate Std. Error z value Pr(>|z|)  
(Intercept)                             -5.7620   343.5993  -0.017   0.9866  
health.L                                -9.3791   921.9735  -0.010   0.9919  
health.Q                                -7.1542   687.1985  -0.010   0.9917  
health.C                                -3.2635   307.3247  -0.011   0.9915  
prior_piloryOther                       -0.3021     0.7859  -0.384   0.7007  
prior_piloryI don’t know               -14.3531   979.0393  -0.015   0.9883  
prior_piloryDidn't answer the question  -0.7303     0.3245  -2.250   0.0244 *
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 322.85  on 696 

## Logistic Model

In [39]:
# Fit logistic regression
model_participation <- stats::glm(
  participation ~ gender + education + family_history + knowing + awareness_pilory + awareness_GC, # + prior_pilory + health,
  data = df1,
  family = stats::binomial(link = "logit"),
  contrasts = list(
    gender = "contr.treatment",
    education = "contr.treatment",
    family_history = "contr.treatment",
    knowing = "contr.treatment",
    awareness_pilory = "contr.treatment",
    awareness_GC = "contr.treatment"
    # prior_pilory = "contr.treatment"
    # health = "contr.poly"
  )
)

# Model summary
summary(model_participation)


Call:
stats::glm(formula = participation ~ gender + education + family_history + 
    knowing + awareness_pilory + awareness_GC, family = stats::binomial(link = "logit"), 
    data = df1, contrasts = list(gender = "contr.treatment", 
        education = "contr.treatment", family_history = "contr.treatment", 
        knowing = "contr.treatment", awareness_pilory = "contr.treatment", 
        awareness_GC = "contr.treatment"))

Coefficients:
                                            Estimate Std. Error z value
(Intercept)                                  -4.2354     0.4739  -8.936
genderMale                                    0.2566     0.3484   0.737
educationHigh school or vocational            1.8279     0.3557   5.139
educationPrimary/did not finish high school   2.0276     0.9377   2.162
family_historyNo                              0.6614     0.3640   1.817
family_historyI don't know                    1.1673     0.5818   2.006
knowingYes                                   -0.429

In [40]:
results_participation <- broom::tidy(
  model_participation,
  exponentiate = TRUE,
  conf.int = TRUE
) |>
  dplyr::select(
    term,
    estimate,
    conf.low,
    conf.high,
    p.value
  ) |>
  dplyr::rename(
    Variable = term,
    OR = estimate,
    CI_lower = conf.low,
    CI_upper = conf.high,
    p_value = p.value
  )

results_participation

Variable,OR,CI_lower,CI_upper,p_value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),0.01447424,0.005352703,0.03455362,4.022120e-19
genderMale,1.29249251,0.647934561,2.55925229,4.614075e-01
educationHigh school or vocational,6.22080274,3.091965184,12.56488296,2.760206e-07
educationPrimary/did not finish high school,7.59619930,0.944392469,43.98319443,3.058563e-02
family_historyNo,1.93753103,0.954484897,4.01818154,6.919011e-02
family_historyI don't know,3.21336899,0.966187970,9.71775709,4.482411e-02
knowingYes,0.65070838,0.297185144,1.37290140,2.674632e-01
knowingI don’t know,0.75202520,0.197596950,2.25528970,6.393206e-01
awareness_piloryNo,3.41501960,1.597397622,7.61700887,1.917324e-03


In [41]:
gt_table = results_participation |>
  gt::gt() |>
  gt::tab_header(
    title = "Logistic regression: Participation",
    subtitle = "Odds ratios and 95% confidence intervals"
  ) |>
  gt::cols_label(
    Variable = "Variable",
    OR = "Odds ratio",
    CI_lower = "95% CI (lower)",
    CI_upper = "95% CI (upper)",
    p_value = "p-value"
  ) |>
  gt::fmt_number(
    columns = c(OR, CI_lower, CI_upper),
    decimals = 2
  ) |>
  gt::fmt_number(
    columns = p_value,
    decimals = 3
  ) |>
  gt::tab_options(
    table.font.size = gt::px(13)
  )

# htmltools::HTML(gt::as_raw_html(gt_table))
IRdisplay::display_html(gt::as_raw_html(gt_table))

<table class="gt_table" data-quarto-disable-processing="false" data-quarto-bootstrap="false" style="-webkit-font-smoothing: antialiased; -moz-osx-font-smoothing: grayscale; font-family: system-ui, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif, 'Apple Color Emoji', 'Segoe UI Emoji', 'Segoe UI Symbol', 'Noto Color Emoji'; display: table; border-collapse: collapse; line-height: normal; margin-left: auto; margin-right: auto; color: #333333; font-size: 13px; font-weight: normal; font-style: normal; background-color: #FFFFFF; width: auto; border-top-style: solid; border-top-width: 2px; border-top-color: #A8A8A8; border-right-style: none; border-right-width: 2px; border-right-color: #D3D3D3; border-bottom-style: solid; border-bottom-width: 2px; border-bottom-color: #A8A8A8; border-left-style: none; border-left-width: 2px; border-left-color: #D3D3D3;" bgcolor="#FFFFFF">
 
 
 Logistic regression: Participation 
 
 
 <td colspan="5" class="gt_heading gt_subtitle gt_font_normal gt_bottom_border" style="border-style: none; color: #333333; font-size: 85%; padding-top: 3px; padding-bottom: 5px; padding-left: 5px; padding-right: 5px; border-top-color: #FFFFFF; border-top-width: 0; background-color: #FFFFFF; text-align: center; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; border-bottom-style: solid; border-bottom-width: 2px; border-bottom-color: #D3D3D3; font-weight: normal;" bgcolor="#FFFFFF" align="center">Odds ratios and 95% confidence intervals 
 
 
 <th class="gt_col_heading gt_columns_bottom_border gt_left" rowspan="1" colspan="1" scope="col" id="Variable" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: left;" bgcolor="#FFFFFF" valign="bottom" align="left">Variable 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="OR" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; font-variant-numeric: tabular-nums;" bgcolor="#FFFFFF" valign="bottom" align="right">Odds ratio 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="CI_lower" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; font-variant-numeric: tabular-nums;" bgcolor="#FFFFFF" valign="bottom" align="right">95% CI (lower) 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="CI_upper" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; fon

# Study 2

## What the non-participants said

In [52]:
df2 = haven::read_sav("../data/Pilot2.sav") |> 
    add_participation() |> 
    dplyr::filter(participation %in% c("Didn't participate","Didn't answer the question"))

# Source column names
variables <- c(paste0("G06Q24_SQ", sprintf("%03d", 1:6)), "G06Q24_other")

# Descriptive labels
labels <- c(
  "Concern about the possible discomfort associated with the procedures",
  "Concern about possible health risks",
  "Having to travel too far to take part in procedures",
  "Financial reasons",
  "Belief in my good health and the unnecessity of these procedures",
  "Mistrust towards the healthcare system/ my healthcare provider",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df2$G06Q24_other[
  !is.na(df2$G06Q24_other) &
    trimws(df2$G06Q24_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df2[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why wouldn’t you accept to participate in upper endoscopy and colonoscopy procedures?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why wouldn’t you accept to participate in upper endoscopy and colonoscopy procedures?

,Concern about the possible discomfort associated with the procedures,Other,Concern about possible health risks,Belief in my good health and the unnecessity of these procedures,Having to travel too far to take part in procedures,Mistrust towards the healthcare system/ my healthcare provider,Financial reasons
Concern about the possible discomfort associated with the procedures,69,18,25,16,2,2,0
Other,18,41,12,9,1,6,0
Concern about possible health risks,25,12,39,0,2,3,0
Belief in my good health and the unnecessity of these procedures,16,9,0,39,0,1,0
Having to travel too far to take part in procedures,2,1,2,0,10,0,0
Mistrust towards the healthcare system/ my healthcare provider,2,6,3,1,0,9,0
Financial reasons,0,0,0,0,0,0,0


In [42]:
df2 = haven::read_sav("../data/Pilot2.sav") |> 
    add_participation() |> 
    dplyr::filter(participation %in% c("Didn't participate","Didn't answer the question"))

# Source column names
variables <- c(paste0("G06Q25_SQ", sprintf("%03d", 1:4)), "G06Q25_other")

# Descriptive labels
labels <- c(
  "I am more concerned about my risk of developing colorectal cancer",
  "I find the colonoscopy procedure safer and less invasive than upper endoscopy",
  "Because upper endoscopy tests for stomach cancer and there is no standard screening programme for stomach cancer",
  "I would prefer to undergo these procedures separately",
  "Other"
)

names(labels) <- variables

# Convert any non-empty value in G04Q16_other to "Y"
df2$G06Q25_other[
  !is.na(df2$G06Q25_other) &
    trimws(df2$G06Q25_other) != ""
] <- "Y"

# Extract relevant columns
response_data <- df2[, variables]

# Convert Y to 1 and empty/missing values to 0
response_matrix <- as.matrix(
  as.data.frame(
    lapply(response_data, function(x) {
      as.integer(!is.na(x) & trimws(as.character(x)) == "Y")
    })
  )
)

# Assign descriptive labels
colnames(response_matrix) <- labels

# Calculate pairwise counts
cooccurrence_matrix <- crossprod(response_matrix)

# Get ordering based on diagonal counts (descending)
ordering <- order(
  diag(cooccurrence_matrix),
  decreasing = TRUE
)

# Reorder rows and columns identically
cooccurrence_matrix <- cooccurrence_matrix[
  ordering,
  ordering,
  drop = FALSE
]

# Convert to dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

# Convert to a dataframe for display
cooccurrence_table <- as.data.frame(
  cooccurrence_matrix,
  check.names = FALSE
)

IRdisplay::display_markdown(
  "### Why wouldn’t you accept to participate in the study?"
)

html_table <- knitr::kable(
  cooccurrence_table,
  format = "html",
  align = "c",
  caption = "Number of respondents selecting each pair of reasons"
)

IRdisplay::display_html(
  as.character(html_table)
)

### Why wouldn’t you accept to participate in the study?

,I am more concerned about my risk of developing colorectal cancer,I would prefer to undergo these procedures separately,Other,I find the colonoscopy procedure safer and less invasive than upper endoscopy,Because upper endoscopy tests for stomach cancer and there is no standard screening programme for stomach cancer
I am more concerned about my risk of developing colorectal cancer,36,5,1,2,1
I would prefer to undergo these procedures separately,5,31,1,0,0
Other,1,1,25,0,0
I find the colonoscopy procedure safer and less invasive than upper endoscopy,2,0,0,9,1
Because upper endoscopy tests for stomach cancer and there is no standard screening programme for stomach cancer,1,0,0,1,5


## Data

In [43]:
df2 = haven::read_sav("../data/Pilot2.sav")
df2 = df2 |> 
    add_participation() |>
    add_age() |> dplyr::filter(!(age %in% c("I prefer not to answer", "Didn't answer the question"))) |>
    add_gender() |> dplyr::filter(gender %in% c("Male", "Female")) |>
    add_education() |> dplyr::filter(!(education %in% c("I prefer not to answer", "Didn't answer the question"))) |>
    add_health() |> dplyr::filter(!(health %in% c("Very bad","I don't know","I prefer not to respond","Didn't answer the question"))) |>
    add_family_history(2) |> dplyr::filter(!(family_history %in% c("I prefer not to answer", "Didn't answer the question"))) |>
    add_awareness_GC(2) |> dplyr::filter(!(awareness_GC %in% c("I prefer not to answer", "Didn't answer the question"))) |> 
                                dplyr::mutate(awareness_GC = forcats::fct_collapse(awareness_GC,"No" = c("No", "I don’t know"))) |>
    add_prior_endoscopy() |> dplyr::filter(!(prior_endoscopy %in% c("I prefer not to answer", "Didn't answer the question"))) |>
    add_prior_GC_testing()  |> dplyr::filter(!(prior_endoscopy %in% c("I prefer not to answer", "Didn't answer the question")))

In [44]:
print("----------------prior_GC_testing---------------------")
print(summary(df2$prior_GC_testing))
print("----------------prior_endoscopy---------------------")
print(summary(df2$prior_endoscopy))
print("----------------awareness_GC---------------------")
print(summary(df2$awareness_GC))
print("--------------family_history---------------------")
print(summary(df2$family_history))
print("----------------health---------------------------")
print(summary(df2$health))
print("---------------education-------------------------")
print(summary(df2$education))
print("-----------------gender---------------------------")
print(summary(df2$gender))
print("-----------------age---------------------------")
print(summary(df2$age))
print("----------------participation---------------------")
print(summary(df2$participation))

[1] "----------------prior_GC_testing---------------------"
                        No                      Other 
                       893                         80 
              I don’t know     I prefer not to answer 
                        87                          2 
Didn't answer the question 
                         2 
[1] "----------------prior_endoscopy---------------------"
                       Yes                         No 
                       522                        525 
              I don’t know     I prefer not to answer 
                        17                          0 
Didn't answer the question 
                         0 
[1] "----------------awareness_GC---------------------"
                        No                        Yes 
                       873                        191 
    I prefer not to answer Didn't answer the question 
                         0                          0 
[1] "--------------family_history--------------------

## Linear Model

In [45]:
# Fit linear regression
model_participation <- stats::lm(
  participation ~ age + gender + education + health + family_history + awareness_GC + prior_endoscopy + prior_GC_testing,
  data = df2 |> dplyr::mutate(participation = dplyr::case_when(
      participation == "Participated" ~ 1,
      TRUE ~ 0)),
      contrasts = list(
        age = "contr.poly",
        gender = "contr.treatment",
        education = "contr.treatment",
        health = "contr.poly",
        family_history = "contr.treatment",
        awareness_GC = "contr.treatment",
        prior_endoscopy = "contr.treatment",
        prior_GC_testing = "contr.treatment"
      )
)
# model_participation = broom::tidy(model_participation)

### Coefficients

In [46]:
results_participation <- broom::tidy(
  model_participation,
  exponentiate = FALSE,
  conf.int = TRUE
) |>
  dplyr::select(
    term,
    estimate,
    conf.low,
    conf.high,
    p.value
  ) |>
  dplyr::rename(
    Variable = term,
    CI_lower = conf.low,
    CI_upper = conf.high,
    p_value = p.value
  )

results_participation

Variable,estimate,CI_lower,CI_upper,p_value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),0.966165110,0.91863689,1.013693328,4.031556e-212
age.L,-0.027768079,-0.06913970,0.013603545,1.881181e-01
age.Q,-0.020292755,-0.05941107,0.018825560,3.089528e-01
age.C,0.003715608,-0.03423083,0.041662044,8.476723e-01
age^4,-0.010610671,-0.04668938,0.025468040,5.640029e-01
genderMale,0.019977006,-0.01465263,0.054606647,2.579077e-01
educationHigh school or vocational,-0.036172877,-0.07426310,0.001917350,6.267740e-02
educationPrimary/did not finish high school,0.024338308,-0.03391266,0.082589274,4.124837e-01
health.L,-0.059229687,-0.12568583,0.007226455,8.060830e-02


## Separation Model

In [47]:
model_separation <- stats::glm(
  participation ~ age + gender + education + health + family_history + awareness_GC + prior_endoscopy, #+ prior_GC_testing,
  data = df2,
  family = stats::binomial(link = "logit"),
  contrasts = list(
    age = "contr.poly",
    gender = "contr.treatment",
    education = "contr.treatment",
    health = "contr.poly",
    family_history = "contr.treatment",
    awareness_GC = "contr.treatment",
    prior_endoscopy = "contr.treatment"
    #prior_GC_testing = "contr.treatment"
  ),
  method = detectseparation::detect_separation
)

model_separation

Implementation: ROI | Solver: lpsolve 
Separation: FALSE 
Existence of maximum likelihood estimates
                                (Intercept) 
                                          0 
                                      age.L 
                                          0 
                                      age.Q 
                                          0 
                                      age.C 
                                          0 
                                      age^4 
                                          0 
                                 genderMale 
                                          0 
         educationHigh school or vocational 
                                          0 
educationPrimary/did not finish high school 
                                          0 
                                   health.L 
                                          0 
                                   health.Q 
                                          0 


## Logistic Model

In [48]:
# Fit logistic regression
model_participation <- stats::glm(
  participation ~ age + gender + education + health + family_history + awareness_GC + prior_endoscopy,
  data = df2,
  family = stats::binomial(link = "logit"),
  contrasts = list(
    age = "contr.poly",
    gender = "contr.treatment",
    education = "contr.treatment",
    family_history = "contr.treatment",
    awareness_GC = "contr.treatment",
    prior_endoscopy = "contr.treatment",
    health = "contr.poly"
  )
)

# Model summary
summary(model_participation)


Call:
stats::glm(formula = participation ~ age + gender + education + 
    health + family_history + awareness_GC + prior_endoscopy, 
    family = stats::binomial(link = "logit"), data = df2, contrasts = list(age = "contr.poly", 
        gender = "contr.treatment", education = "contr.treatment", 
        family_history = "contr.treatment", awareness_GC = "contr.treatment", 
        prior_endoscopy = "contr.treatment", health = "contr.poly"))

Coefficients:
                                            Estimate Std. Error z value
(Intercept)                                 -2.90353    0.31951  -9.088
age.L                                        0.47749    0.26119   1.828
age.Q                                        0.28143    0.24975   1.127
age.C                                       -0.08331    0.24789  -0.336
age^4                                        0.07298    0.24073   0.303
genderMale                                  -0.34629    0.22568  -1.534
educationHigh school or vocational

In [49]:
results_participation <- broom::tidy(
  model_participation,
  exponentiate = TRUE,
  conf.int = TRUE
) |>
  dplyr::select(
    term,
    estimate,
    conf.low,
    conf.high,
    p.value
  ) |>
  dplyr::rename(
    Variable = term,
    OR = estimate,
    CI_lower = conf.low,
    CI_upper = conf.high,
    p_value = p.value
  )

results_participation

Variable,OR,CI_lower,CI_upper,p_value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),0.0548291,0.02855512,0.1001723,1.012776e-19
age.L,1.6120269,0.96892431,2.7084426,6.752388e-02
age.Q,1.3250210,0.80740823,2.1559396,2.598195e-01
age.C,0.9200695,0.56162826,1.4906494,7.368203e-01
age^4,1.0757080,0.66786966,1.7233183,7.617733e-01
genderMale,0.7073067,0.45218901,1.0977038,1.249224e-01
educationHigh school or vocational,1.6329856,0.99331643,2.7654385,5.941068e-02
educationPrimary/did not finish high school,0.7776004,0.30916896,1.7932767,5.712473e-01
health.L,1.7661921,0.82900114,4.0881049,1.552258e-01


In [50]:
gt_table = results_participation |>
  gt::gt() |>
  gt::tab_header(
    title = "Logistic regression: Participation",
    subtitle = "Odds ratios and 95% confidence intervals"
  ) |>
  gt::cols_label(
    Variable = "Variable",
    OR = "Odds ratio",
    CI_lower = "95% CI (lower)",
    CI_upper = "95% CI (upper)",
    p_value = "p-value"
  ) |>
  gt::fmt_number(
    columns = c(OR, CI_lower, CI_upper),
    decimals = 2
  ) |>
  gt::fmt_number(
    columns = p_value,
    decimals = 3
  ) |>
  gt::tab_options(
    table.font.size = gt::px(13)
  )

# htmltools::HTML(gt::as_raw_html(gt_table))
IRdisplay::display_html(gt::as_raw_html(gt_table))

<table class="gt_table" data-quarto-disable-processing="false" data-quarto-bootstrap="false" style="-webkit-font-smoothing: antialiased; -moz-osx-font-smoothing: grayscale; font-family: system-ui, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif, 'Apple Color Emoji', 'Segoe UI Emoji', 'Segoe UI Symbol', 'Noto Color Emoji'; display: table; border-collapse: collapse; line-height: normal; margin-left: auto; margin-right: auto; color: #333333; font-size: 13px; font-weight: normal; font-style: normal; background-color: #FFFFFF; width: auto; border-top-style: solid; border-top-width: 2px; border-top-color: #A8A8A8; border-right-style: none; border-right-width: 2px; border-right-color: #D3D3D3; border-bottom-style: solid; border-bottom-width: 2px; border-bottom-color: #A8A8A8; border-left-style: none; border-left-width: 2px; border-left-color: #D3D3D3;" bgcolor="#FFFFFF">
 
 
 Logistic regression: Participation 
 
 
 <td colspan="5" class="gt_heading gt_subtitle gt_font_normal gt_bottom_border" style="border-style: none; color: #333333; font-size: 85%; padding-top: 3px; padding-bottom: 5px; padding-left: 5px; padding-right: 5px; border-top-color: #FFFFFF; border-top-width: 0; background-color: #FFFFFF; text-align: center; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; border-bottom-style: solid; border-bottom-width: 2px; border-bottom-color: #D3D3D3; font-weight: normal;" bgcolor="#FFFFFF" align="center">Odds ratios and 95% confidence intervals 
 
 
 <th class="gt_col_heading gt_columns_bottom_border gt_left" rowspan="1" colspan="1" scope="col" id="Variable" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: left;" bgcolor="#FFFFFF" valign="bottom" align="left">Variable 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="OR" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; font-variant-numeric: tabular-nums;" bgcolor="#FFFFFF" valign="bottom" align="right">Odds ratio 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="CI_lower" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; font-variant-numeric: tabular-nums;" bgcolor="#FFFFFF" valign="bottom" align="right">95% CI (lower) 
 <th class="gt_col_heading gt_columns_bottom_border gt_right" rowspan="1" colspan="1" scope="col" id="CI_upper" style="border-style: none; color: #333333; background-color: #FFFFFF; font-size: 100%; font-weight: normal; text-transform: inherit; border-left-style: none; border-left-width: 1px; border-left-color: #D3D3D3; border-right-style: none; border-right-width: 1px; border-right-color: #D3D3D3; vertical-align: bottom; padding-top: 5px; padding-bottom: 6px; padding-left: 5px; padding-right: 5px; overflow-x: hidden; text-align: right; fon